# Decision Tree — Regression

---

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i, y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

and

$$
y_i \in \mathbb{R}
$$

The goal is to learn a function that partitions the feature space and predicts continuous values.

---

## 2. Model Structure

A decision tree consists of recursive binary splits of the form

$$
x_j \le t
$$

where:

- $j$ = feature index  
- $t$ = threshold  

Each internal node performs a split, and each leaf node outputs a real-valued prediction.

---

## 3. Data Splitting

The dataset is partitioned as:

$$
\mathcal{D}_{\text{left}} = \{(x_i, y_i) : x_{i,j} \le t\}
$$

$$
\mathcal{D}_{\text{right}} = \{(x_i, y_i) : x_{i,j} > t\}
$$

---

## 4. Loss Functions

The model supports three regression losses:

### Mean Squared Error (MSE)

$$
\text{MSE}(y) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \bar{y})^2 = \text{Var}(y)
$$



### Mean Absolute Error (MAE)

$$
\text{MAE}(y) = \frac{1}{N} \sum_{i=1}^{N} |y_i - \tilde{y}|
$$

where:

$$
\tilde{y} = \text{median}(y)
$$



### Huber Loss

Let residuals be:

$$
r_i = y_i - \bar{y}
$$

Then:

$$
L_{\delta}(r_i) =
\begin{cases}
\frac{1}{2} r_i^2 & |r_i| \le \delta \\
\delta |r_i| - \frac{1}{2} \delta^2 & |r_i| > \delta
\end{cases}
$$

Total loss:

$$
\text{Huber}(y) = \sum_{i=1}^{N} L_{\delta}(r_i)
$$

---

## 5. Split Objective

For a split into left and right nodes:

$$
\text{Score} =
\frac{n_L}{n} L(y_{\text{left}}) +
\frac{n_R}{n} L(y_{\text{right}})
$$

where:

- $n_L$, $n_R$ are number of samples  
- $L(\cdot)$ is chosen loss (MSE / MAE / Huber)  

---

## 6. Feature Subsampling

A subset of features is selected:

$$
\mathcal{F} \subset \{1,2,\dots,D\}
$$

with size:

$$
|\mathcal{F}| = \max\left(1, \lfloor D \cdot \text{feature\_fraction} \rfloor \right)
$$

---

## 7. Threshold Selection

For each selected feature, candidate thresholds are computed as:

$$
t_k = \frac{x_{(k)} + x_{(k+1)}}{2}
$$

where $x_{(k)}$ are sorted unique values.

---

## 8. Leaf Node Prediction

Prediction at a leaf node depends on the loss:

- For MSE and Huber Loss:

$$
\hat{y} = \bar{y} = \frac{1}{N} \sum y_i
$$

- For MAE:

$$
\hat{y} = \text{median}(y)
$$

---

## 9. Stopping Criteria

Tree growth stops if:

$$
\text{depth} \ge \text{max\_depth}
$$

or

$$
N < \text{min\_samples\_split}
$$

or

$$
\text{Var}(y) = 0
$$

---

## 10. Best Split Selection

At each node:

$$
(j^*, t^*) =
\arg\min_{j \in \mathcal{F},\, t} \text{Score}(j,t)
$$

---

## 11. Recursive Tree Construction

1. Check stopping condition  
2. Find best feature and threshold  
3. Split dataset  
4. Recursively build left and right subtrees  

---

## 12. Prediction Rule

For a new sample $x$:

$$
\text{if } x_j \le t \rightarrow \text{left subtree}
$$

$$
\text{else } \rightarrow \text{right subtree}
$$

Final prediction:

$$
\hat{y} = \text{leaf value}
$$

---

## 13. Algorithm Summary

At each node:

$$
\text{Select feature subset } \mathcal{F}
$$

$$
\text{Compute thresholds}
$$

$$
(j^*, t^*) = \arg\min \text{Score}
$$

$$
\text{Split data}
$$

Repeat recursively until stopping condition is met.

---

## 14. Final Optimization Perspective

The model greedily minimizes:

$$
\min_{\text{tree}} \sum_{\text{nodes}} \left(
\frac{n_L}{n} L(y_L) + \frac{n_R}{n} L(y_R)
\right)
$$

This produces a piecewise constant function that approximates the target variable.

In [1]:
class LeafNode:
    """
    Leaf node of the decision tree.

    Parameters
    ----------
    value : float
        The predicted value at the leaf node (mean or median of targets).
    """
    def __init__(self, value):
        self.value = value

In [2]:
class DecisionNode:
    """
    Internal decision node of the tree.

    Parameters
    ----------
    best_feature : int
        Index of the feature used for splitting.
    best_threshold : float
        Threshold value for the split.
    left_child : Node
        Left subtree (samples satisfying condition).
    right_child : Node
        Right subtree (samples not satisfying condition).
    """
    def __init__(self, best_feature , best_threshold , left_child , right_child):
        self.best_feature = best_feature
        self.best_threshold = best_threshold
        self.left_child = left_child
        self.right_child = right_child

In [3]:
class DecisionTreeRegressor:
    """
    Decision Tree Regressor supporting MSE, MAE, and Huber loss.

    Parameters
    ----------
    max_depth : int, default=10
        Maximum depth of the tree.

    feature_fraction : float, default=1.0
        Fraction of features randomly selected at each split.

    scoring : str, default='mse'
        Loss function used for splitting.
        Options: 'mse', 'mae', 'huber'

    min_sample_split : int, default=1
        Minimum number of samples required to split.

    delta : float, default=0.5
        Threshold parameter for Huber loss.

    Attributes
    ----------
    root : Node
        Root node of the decision tree.
    """
    def __init__(self, max_depth = 10 , feature_fraction = 1 , scoring='mse', min_sample_split=1 , delta=0.5):
        self.max_depth = max_depth
        self.feature_fraction = feature_fraction
        self.scoring = scoring
        self.min_sample_split = min_sample_split
        self.delta = delta
        self.root = None

        # Validate
        if type(self.max_depth) != int or self.max_depth <= 0:
            raise ValueError('Max depth must be a positive integer')
        if type(self.min_sample_split) != int or self.min_sample_split <= 0:
            raise ValueError('Min samples split must be a positive integer')
        if self.scoring not in ['mse', 'mae','huber']:
            raise ValueError("Scoring must be either 'mse', 'mae' or 'huber'")   
        if not (0 < self.feature_fraction <= 1):
            raise ValueError("feature_fraction must be in the range (0, 1].")
        if self.delta <= 0:
            raise ValueError("delta must be strictly greater than 0.")
            
    def _random_feature_sampling(self, data):
        """
        Randomly select a subset of features.

        Returns
        -------
        selected_features : array-like
            Indices of selected features.
        """
        num_features = data.shape[1]-1 # exclude target column
        total_features_selected = max(1,round(self.feature_fraction * num_features ))
        selected_features = np.random.choice(num_features,total_features_selected,replace=False)

        return selected_features

    def _stopping_condition(self,data,depth):
        """
        Check whether to stop splitting.

        Stops if:
        - Maximum depth reached
        - Not enough samples
        - Target variance is zero
        """
        if depth >= self.max_depth:
            return True
        if len(data) < self.min_sample_split:
            return True
        if np.var(data[:,-1])==0:
            return True
        return False

    def _leaf_prediction(self,y):
        """
        Compute prediction for a leaf node.

        Returns mean (MSE/HUBER) or median (MAE).
        """
        if self.scoring == 'mae':
            return np.median(y)
        else :
            return np.mean(y)
    
    def _all_thresholds(self, data, selected_features):
        """
        Generate candidate thresholds for each selected feature.
        """
        all_thresholds = []
        for feature in selected_features:
            unique_values = np.unique(data[:,feature])
            if len(unique_values) ==1:
                all_thresholds.append(np.array([]))
            else :
                # Midpoints between consecutive sorted values
                unique_averages = (unique_values[1:] + unique_values[:-1])/2
                all_thresholds.append(unique_averages)

        return all_thresholds

    def _split(self,data,feature , threshold):
        """
        Split dataset based on feature and threshold.
        """
        condition = data[:,feature] <= threshold

        return data[condition] , data[~condition]

    def _mse_loss(self,y):
        """Mean Squared Error (variance)."""
        return np.var(y)

    def _mae_loss(self,y):
        """Mean Absolute Error."""
        return np.mean(np.abs(y-np.median(y)))

    def _huber_loss(self,y):
        """Huber loss (robust to outliers)."""
        r = y- np.mean(y)
        r_abs = np.abs(r)

        return np.sum(np.where(r_abs <= self.delta ,(0.5*r*r), self.delta * r_abs - 0.5* (self.delta**2)))

    def _score(self, y_left , y_right):
        """
        Compute weighted loss for a split.
        """
        n_left = len(y_left)
        n_right = len(y_right)

        if n_left == 0 or n_right==0:
            return np.inf

        total = n_left + n_right
        if self.scoring =='mse':
            left_score = self._mse_loss(y_left) 
            right_score = self._mse_loss(y_right) 


        elif self.scoring =='mae':
            left_score = self._mae_loss(y_left)
            right_score = self._mae_loss(y_right)

        elif self.scoring =='huber':
            left_score = self._huber_loss(y_left)
            right_score = self._huber_loss(y_right)  

        return (n_left*left_score + n_right*right_score) / total

    def _find_best_feature_threshold(self,data,all_thresholds,selected_features):
        """
        Find the best feature and threshold minimizing split loss.
        """
        best_feature = None
        best_threshold = None
        best_score = np.inf

        for i ,  feature in enumerate(selected_features):
            thresholds = all_thresholds[i]
            if len(thresholds) ==0 :
                continue
            
            else:
                for threshold in thresholds:
                    left , right = self._split(data,feature ,threshold)
                    feature_score = self._score(left[:,-1] , right[:,-1])

                    if feature_score < best_score:
                        best_score = feature_score
                        best_feature = feature
                        best_threshold = threshold


        return best_feature , best_threshold

    def _find_best_split(self,data):
        """
        Select features and compute best split.
        """
        selected_features = self._random_feature_sampling(data)

        all_thresholds = self._all_thresholds(data, selected_features)

        best_feature , best_threshold = self._find_best_feature_threshold(data,all_thresholds,selected_features)

        return best_feature , best_threshold


    def _build_tree(self,data,depth):
        """
        Recursively build the decision tree.
        """
        if self._stopping_condition(data,depth):
            return LeafNode(self._leaf_prediction(data[:,-1]))

        best_feature , best_threshold = self._find_best_split(data)

        if best_feature is None:
            return LeafNode(self._leaf_prediction(data[:,-1]))

        left_data , right_data = self._split(data,best_feature , best_threshold)

        left_child = self._build_tree(left_data,depth+1)
        right_child = self._build_tree(right_data,depth+1)


        return DecisionNode(best_feature , best_threshold , left_child , right_child)

    def fit(self,X,y):
        """
        Train the decision tree.

        Parameters
        ----------
        X : array-like of shape (N, D)
            Feature matrix

        y : array-like of shape (N,)
            Target values
        """
        X = np.asarray(X)
        y = np.asarray(y)

        y = y.reshape(-1)


        if X.ndim==1:
            X = X.reshape(-1,1)
        data = np.column_stack((X,y))

        self.root = self._build_tree(data , depth=0)


    def _predict_single(self,x,node):
        """
        Predict output for a single sample.
        """
        if isinstance(node, LeafNode):
            return node.value
        if isinstance(node, DecisionNode):
            if x[node.best_feature] <= node.best_threshold:
                return self._predict_single(x, node.left_child)
            else:
                return self._predict_single(x, node.right_child)

    def predict(self, X):
        """
        Predict outputs for input samples.

        Parameters
        ----------
        X : array-like of shape (N, D)

        Returns
        -------
        predictions : ndarray
        """
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return np.array([self._predict_single(x, self.root) for x in X])

---

## 1. Dataset Construction

Construct a simple dataset of 50 points with a linear trend:

$$
y = x + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 2)
$$

Add 3 extreme outliers (+30) to see the effect of different loss functions.

---

## 2. Decision Tree Loss Functions

Train four trees (max depth = 2) using different losses:

1. **MSE Tree**: uses mean squared error (sensitive to outliers)  
2. **MAE Tree**: uses mean absolute error (robust to outliers)  
3. **Huber $\delta=0.1$**: behaves similarly to MAE (robust to outliers)  
4. **Huber $\delta=50$**: behaves similarly to MSE (sensitive to outliers)  

The Huber loss interpolates between MSE and MAE depending on $\delta$:

$$
L_\delta(r) = 
\begin{cases} 
0.5 r^2 & \text{if } |r| \le \delta \\
\delta (|r| - 0.5\delta) & \text{if } |r| > \delta
\end{cases}
$$

---

In [4]:
import numpy as np
import pandas as pd

np.random.seed(42)
X = np.arange(50).reshape(-1,1)
y = X.flatten() + np.random.normal(0, 2, size=50)  # linear with small noise

# Add  outliers
outlier_indices = np.random.choice(np.arange(50), size=3, replace=False)
y[outlier_indices] += 30


In [5]:
# 2.All models
trees = {
    "MSE": DecisionTreeRegressor(max_depth=2, scoring='mse'),
    "MAE": DecisionTreeRegressor(max_depth=2, scoring='mae'),
    "Huber δ=.1": DecisionTreeRegressor(max_depth=2, scoring='huber', delta=.1),   # behaves more like MAE
    "Huber δ=50": DecisionTreeRegressor(max_depth=2, scoring='huber', delta=50), # behaves more like MSE
}

In [6]:

# 3. Train and predict
results = []

def mse(y_true, y_pred): return np.mean((y_true - y_pred)**2)
def mae(y_true, y_pred): return np.mean(np.abs(y_true - y_pred))

for name, tree in trees.items():
    tree.fit(X, y)
    y_pred = tree.predict(X)
    
    # Only show the relevant loss for clarity
    if 'MSE' in name or 'δ=50' in name:
        mse_val = round(mse(y, y_pred),2)
        mae_val = '-'
    elif 'MAE' in name or 'δ=.1' in name:
        mse_val = '-'
        mae_val = round(mae(y, y_pred),2)
    
    results.append({
        "Model": name,
        "MSE Loss": mse_val,
        "MAE Loss": mae_val
    })


df_results = pd.DataFrame(results)

---
## 3. Results Table



In [7]:
df_results

,Model,MSE Loss,MAE Loss
0,MSE,48.8,-
1,MAE,-,4.31
2,Huber δ=.1,-,4.52
3,Huber δ=50,52.53,-


---
## 4. Observations

- **MSE Tree**: Leaf prediction is strongly affected by outliers.  
- **MAE Tree**: Leaf prediction is robust.  
- **Huber $\delta=0.1$**: Approximates MAE → robust to outliers.  
- **Huber $\delta=50$**: Approximates MSE → sensitive to outliers.  
---

## 5. Conclusion 
Huber loss provides a tunable trade-off between sensitivity to outliers and robustness by adjusting $\delta$.

---